# Load and clean all property data ready for analysis:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import duckdb
import seaborn as sns
from esda.moran import Moran
from spreg import OLS, ML_Error, ML_Lag, GM_Error, GM_Lag
from splot.esda import moran_scatterplot, plot_moran
import contextily as ctx
from matplotlib.ticker import FuncFormatter
from libpysal.weights import lag_spatial
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.cm as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.patches import Patch
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

import libpysal as ps 
from libpysal.weights import Queen

In [2]:
# Creates .db file in the project folder
conn = duckdb.connect("project.db")

# Install and load the spatial extension
conn.execute("INSTALL spatial;")
conn.execute("LOAD spatial;")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Create schemas to organize raw and clean data
conn.execute("CREATE SCHEMA IF NOT EXISTS raw_data;")
conn.execute("CREATE SCHEMA IF NOT EXISTS model_data;")

,database,schema,name,column_names,column_types,temporary


In [ ]:
# Residential transactions data:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw_data.ppd_transactions AS
    SELECT 
        postcode,
        transactionid,
        price,
        dateoftransfer,
        propertytype,
        oldnew,
        duration,
        towncity,
        district,
        county,
        categorytype,
        lad23cd,
        CURRENT_ENERGY_RATING,
        PROPERTY_TYPE,
        BUILT_FORM,
        LOCAL_AUTHORITY_LABEL,
        CONSTITUENCY_LABEL,
        POSTTOWN,
        CONSTRUCTION_AGE_BAND,
        UPRN,
        TOTAL_FLOOR_AREA,
        priceper
    FROM read_csv_auto(
        'tranall_link_26122024.csv',
        strict_mode=false,
        max_line_size=10000000,
        delim=',',
        quote='"',
        escape='"',
        header=true
    )
""")

print(f"Transactions loaded: {conn.execute('SELECT COUNT(*) FROM raw_data.ppd_transactions').fetchone()[0]:,}")

In [ ]:
# OS Unique Property Reference Number (UPRN) data:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw_data.os_uprn AS
    SELECT *
    FROM read_csv_auto(
        'osopenuprn_202606.csv',
        strict_mode=false,
        max_line_size=10000000,
        delim=',',
        quote='"',
        escape='"',
        header=true
    )
""")

print(f"UPRNs loaded: {conn.execute('SELECT COUNT(*) FROM raw_data.os_uprn').fetchone()[0]:,}")

In [ ]:
# Postcode to LSOA lookup:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw_data.postcode_to_lsoa AS
    SELECT * 
    FROM read_csv_auto(
        'postcode_LSOA.csv',
        header=true,
        delim=',',
        quote='"',
        escape='"')
""")

print(f"Postcodes to LSOAs loaded: {conn.execute('SELECT COUNT(*) FROM raw_data.postcode_to_lsoa').fetchone()[0]:,}")

In [ ]:
# load UPRN information:
uprn_data = conn.execute("SELECT * FROM raw_data.os_uprn").df()

# Load postcode to lsoa lookup:
postcode_data = conn.execute("SELECT * FROM raw_data.postcode_to_lsoa").df()

In [ ]:
# Load LM PPD transactions data:
ppd_transactions = conn.execute("SELECT * FROM raw_data.ppd_transactions").df()
ppd_transactions.head()